# ETL — V1DD 1196 synapses (`SynapseConnectivityLong` + `SynapseFeatureMatrix`)

Registers the V1DD `1196` synapse table into the two new synapse classes: the long
single-synapse connectivity table and a wide per-synapse feature matrix
(position, size, `synaptictargetlabel`). **Identifiers:** `project_id="v1dd"`,
`dataset_id="v1dd_1196"`, `feature_matrix_id="v1dd_1196_synapse_features"`.

This notebook is a **sampled digest**: it writes a small slice with
`write_deltalake` directly (there is no `WriteSpec` for these classes yet — see
the *IO write layer* note below) into an isolated `DEMO_ROOT`, then reads it
back with `read_synapse_table`.

In [1]:
## for temp notetaking 
from IPython.display import HTML
HTML("""
<style>
.yynote { color: red; font-weight: bold; font-size: 12pt; }
</style>
""")

<span class="yynote">I will leave comments</span>

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.models import SynapseConnectivityLong, SynapseFeatureMatrix
from connects_common_connectivity.io import read_synapse_table
from connects_common_connectivity.io.arrow_utils import (
    build_arrow_schema,
    models_to_table,
    attach_linkml_metadata,
)

In [2]:
# --- Constants -------------------------------------------------------------
DATA_ROOT = Path("/data/v1dd_1196")
SYN_FEATHER = DATA_ROOT / "syn_df_all_to_proofread_to_all_1196.feather"
LABEL_FEATHER = DATA_ROOT / "syn_label_df_all_to_proofread_to_all_1196.feather"

PROJECT_ID = "v1dd"
DATASET_ID = "v1dd_1196"
FEATURE_MATRIX_ID = "v1dd_1196_synapse_features"

# Feature columns that go into the wide per-synapse feature Parquet.
# Position + size come from syn_df; synaptictargetlabel is LEFT-joined from the
# label feather and is null for synapses without a label.
POSITION_COLS = [
    "pre_pt_position_x", "pre_pt_position_y", "pre_pt_position_z",
    "post_pt_position_x", "post_pt_position_y", "post_pt_position_z",
    "ctr_pt_position_x", "ctr_pt_position_y", "ctr_pt_position_z",
]
FEATURE_COLS = POSITION_COLS + ["size", "synaptictargetlabel"]

# Sampled digest: keep the notebook fast and non-destructive.
SAMPLE_N = 20_000

# Production writes would target get_settings().output_root directly once the
# WriteSpec entries below exist. Until then we isolate the digest under a
# dedicated demo root so nothing in the shared lake is touched.
OUTPUT_ROOT = get_settings().output_root
DEMO_ROOT = OUTPUT_ROOT / "_synapse_digest_v1dd_1196"

for k, v in dict(DATA_ROOT=DATA_ROOT, OUTPUT_ROOT=OUTPUT_ROOT, DEMO_ROOT=DEMO_ROOT,
                 PROJECT_ID=PROJECT_ID, DATASET_ID=DATASET_ID,
                 FEATURE_MATRIX_ID=FEATURE_MATRIX_ID, SAMPLE_N=SAMPLE_N).items():
    print(f"{k} = {v}")

DATA_ROOT = /data/v1dd_1196
OUTPUT_ROOT = /root/capsule/scratch/em_patchseq_wnm_v2
DEMO_ROOT = /root/capsule/scratch/em_patchseq_wnm_v2/_synapse_digest_v1dd_1196
PROJECT_ID = v1dd
DATASET_ID = v1dd_1196
FEATURE_MATRIX_ID = v1dd_1196_synapse_features
SAMPLE_N = 20000


<span class="yynote">output root wrong, demo root name inconsistent w</span>

In [3]:
# --- Prerequisite check ----------------------------------------------------
for p in (SYN_FEATHER, LABEL_FEATHER):
    assert p.exists(), f"Missing source feather: {p}"
print("source feathers present")

source feathers present


## Load source

`syn_df` is one row per synapse (`id` unique); `syn_label_df` gives a
spine/shaft/soma `tag` for a **subset** of synapses (renamed here to
`synaptictargetlabel`). Root ids are 18-digit ints kept as strings.

In [4]:
syn_df = pd.read_feather(SYN_FEATHER)
label_df = (
    pd.read_feather(LABEL_FEATHER)
    .reset_index()                                  # 'id' is the feather index
    .rename(columns={"tag": "synaptictargetlabel"})
)
print("syn_df:", syn_df.shape)
print("label_df:", label_df.shape,
      "| labels cover", f"{label_df['id'].nunique()/len(syn_df):.1%}", "of synapses")
syn_df.head(3)

syn_df: (8204497, 13)


label_df: (6706286, 2) | labels cover 81.7% of synapses


,id,pre_pt_position_x,pre_pt_position_y,pre_pt_position_z,post_pt_position_x,post_pt_position_y,post_pt_position_z,ctr_pt_position_x,ctr_pt_position_y,ctr_pt_position_z,size,pre_pt_root_id,post_pt_root_id
0,354386968,758200.5,802316.1,304380.0,757861.0,802558.6,304650.0,757967.7,802597.4,304380.0,240,864691132536286810,864691132734919083
1,378070488,792063.2,514342.5,183735.0,792664.6,514284.3,183915.0,792412.4,514294.0,183735.0,3056,864691132572190492,864691132606767301
2,499493001,977071.3,390075.8,191340.0,976974.3,390104.9,190935.0,976838.5,390337.7,190935.0,1346,864691132573738810,864691132747578447


In [5]:
# Sample for the digest; ids stay strings (never cast).
sample = syn_df.head(SAMPLE_N).copy()
sample["id"] = sample["id"].astype(str)
sample["pre_pt_root_id"] = sample["pre_pt_root_id"].astype(str)
sample["post_pt_root_id"] = sample["post_pt_root_id"].astype(str)
print("sample:", sample.shape)
sample[["id", "pre_pt_root_id", "post_pt_root_id", "size"]].head(3)

sample: (20000, 13)


,id,pre_pt_root_id,post_pt_root_id,size
0,354386968,864691132536286810,864691132734919083,240
1,378070488,864691132572190492,864691132606767301,3056
2,499493001,864691132573738810,864691132747578447,1346


## Schema mapping

- **`SynapseConnectivityLong`** — `id` (synapse id), `presynaptic_cell` /
  `postsynaptic_cell` (DataItem root ids), `dataset_id`, `project_id`. One row =
  one synapse; the pre/post pair is intentionally not unique.
- **`SynapseFeatureMatrix`** — pointer to the wide feature Parquet keyed by the
  synapse id, joined LEFT onto the long table.

### Write 1 — long single-synapse table → `synapse/`

Built as `SynapseConnectivityLong` models → Arrow (with LinkML metadata) →
`write_deltalake`, partitioned by `project_id` and overwrite-scoped to
`(project_id, dataset_id)`.

In [6]:
long_rows = [
    SynapseConnectivityLong(
        id=r.id,
        presynaptic_cell=r.pre_pt_root_id,
        postsynaptic_cell=r.post_pt_root_id,
        dataset_id=DATASET_ID,
        project_id=PROJECT_ID,
    )
    for r in sample.itertuples()
]
long_schema = build_arrow_schema(SynapseConnectivityLong)
long_table = attach_linkml_metadata(
    models_to_table(long_rows, schema=long_schema),
    linkml_class="SynapseConnectivityLong",
)
write_deltalake(
    str(DEMO_ROOT / "synapse"),
    long_table,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND dataset_id = '{DATASET_ID}'",
    partition_by=["project_id"],
)
print("long rows written:", long_table.num_rows)

long rows written: 20000


In [7]:
# verify — long table reads back with the expected scope and columns
_long = read_synapse_table(PROJECT_ID, dataset_id=DATASET_ID, output_root=DEMO_ROOT)
print("shape:", _long.shape)
assert _long.height == len(long_rows)
assert _long["id"].n_unique() == _long.height          # synapse id is the PK
print(_long.head(3))

shape: (20000, 5)
shape: (3, 5)
┌───────────┬────────────────────┬────────────────────┬────────────┬────────────┐
│ id        ┆ presynaptic_cell   ┆ postsynaptic_cell  ┆ dataset_id ┆ project_id │
│ ---       ┆ ---                ┆ ---                ┆ ---        ┆ ---        │
│ str       ┆ str                ┆ str                ┆ str        ┆ str        │
╞═══════════╪════════════════════╪════════════════════╪════════════╪════════════╡
│ 354386968 ┆ 864691132536286810 ┆ 864691132734919083 ┆ v1dd_1196  ┆ v1dd       │
│ 378070488 ┆ 864691132572190492 ┆ 864691132606767301 ┆ v1dd_1196  ┆ v1dd       │
│ 499493001 ┆ 864691132573738810 ┆ 864691132747578447 ┆ v1dd_1196  ┆ v1dd       │
└───────────┴────────────────────┴────────────────────┴────────────┴────────────┘


### Write 2 — wide per-synapse feature Parquet → `synapsefeatures/<id>/`

The wide table is built from raw dataframes (not model instances), keyed by the
synapse `id`, with `synaptictargetlabel` LEFT-joined from the label feather.
This mirrors how `cellfeatures/` wide Parquets are handled — outside the model
registry.

In [8]:
wide = sample[["id"] + POSITION_COLS + ["size"]].merge(
    label_df[["id", "synaptictargetlabel"]].astype({"id": str}),
    on="id", how="left",
)
wide["project_id"] = PROJECT_ID
wide["dataset_id"] = DATASET_ID
write_deltalake(
    str(DEMO_ROOT / "synapsefeatures" / FEATURE_MATRIX_ID),
    pa.Table.from_pandas(wide, preserve_index=False),
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id"],
)
print("feature rows written:", len(wide),
      "| labelled:", wide["synaptictargetlabel"].notna().sum())

feature rows written: 20000 | labelled: 9480


In [9]:
# verify — join features onto the long table; unlabeled synapses -> null
_full = read_synapse_table(
    PROJECT_ID, dataset_id=DATASET_ID,
    features=True, feature_matrix_id=FEATURE_MATRIX_ID,
    output_root=DEMO_ROOT,
)
print("shape:", _full.shape)
assert _full.height == len(long_rows)                  # LEFT join preserves rows
assert "synaptictargetlabel" in _full.columns and "ctr_pt_position_x" in _full.columns
print("null labels:", _full["synaptictargetlabel"].null_count())
_full.select(["id", "presynaptic_cell", "postsynaptic_cell", "size", "synaptictargetlabel"]).head(3)

shape: (20000, 16)
null labels: 10520


id,presynaptic_cell,postsynaptic_cell,size,synaptictargetlabel
str,str,str,i64,str
"""354386968""","""864691132536286810""","""864691132734919083""",240,"""shaft"""
"""378070488""","""864691132572190492""","""864691132606767301""",3056,"""shaft"""
"""499493001""","""864691132573738810""","""864691132747578447""",1346,null


In [10]:
# verify — subset feature selection appends only the requested columns
_sub = read_synapse_table(
    PROJECT_ID, dataset_id=DATASET_ID,
    features=["size", "synaptictargetlabel"], feature_matrix_id=FEATURE_MATRIX_ID,
    output_root=DEMO_ROOT,
)
assert "ctr_pt_position_x" not in _sub.columns and "size" in _sub.columns
print("subset columns:", _sub.columns)

subset columns: ['id', 'presynaptic_cell', 'postsynaptic_cell', 'dataset_id', 'project_id', 'size', 'synaptictargetlabel']


### Write 3 — `SynapseFeatureMatrix` pointer row → `synapsefeaturematrix/`

A single metadata row recording where the wide feature Parquet lives and which
column is the synapse id.

In [11]:
pmm = SynapseFeatureMatrix(
    id=FEATURE_MATRIX_ID,
    description="Per-synapse position, size and target label for V1DD 1196.",
    dataset_id=DATASET_ID,
    project_id=PROJECT_ID,
    parquet_path=f"file://{(DEMO_ROOT / 'synapsefeatures' / FEATURE_MATRIX_ID).resolve()}/",
    synapse_index_column="id",
)
fm_schema = build_arrow_schema(SynapseFeatureMatrix)
fm_table = attach_linkml_metadata(
    models_to_table([pmm], schema=fm_schema),
    linkml_class="SynapseFeatureMatrix",
)
write_deltalake(
    str(DEMO_ROOT / "synapsefeaturematrix"),
    fm_table,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND id = '{FEATURE_MATRIX_ID}'",
    partition_by=["project_id"],
)
print("feature-matrix pointer rows written:", fm_table.num_rows)

feature-matrix pointer rows written: 1


## IO write-layer note — out of scope here, TODO

I deliberately did **not** touch `io/write_spec.py` or `io/writers.py`. To make
these classes first-class `write_models(...)` citizens later, add two
`WriteSpec` entries to `io/write_spec.py` (and import the classes there):

```python
"SynapseConnectivityLong": WriteSpec(
    model_cls=SynapseConnectivityLong,
    subdir="synapse",
    partition_by=["project_id"],
    scope_columns=["project_id", "dataset_id"],   # rewrite one dataset slice
    write_mode="overwrite_scoped",
    required_for_write=["dataset_id"],
),
"SynapseFeatureMatrix": WriteSpec(
    model_cls=SynapseFeatureMatrix,
    subdir="synapsefeaturematrix",
    partition_by=["project_id"],
    scope_columns=["project_id", "id"],
    write_mode="overwrite_scoped",
),
```

Notes: `writers.py` needs no change — its generic dispatch already handles any
registered spec. The wide `synapsefeatures/<id>/` Parquet stays **outside** the
registry (built from raw dataframes, exactly like `cellfeatures/`). For huge
datasets an incremental `append_new_by_id` mode keyed on the unique synapse `id`
may be preferable to `overwrite_scoped`; that is a mode decision to make when the
schema/writer work is picked up. Once the specs exist, the three `write_deltalake`
calls above collapse to `write_models(long_rows)` / `write_models([pmm])`, and the
digest can target `get_settings().output_root` instead of `DEMO_ROOT`.

## Summary

| Output (under `DEMO_ROOT`) | Rows |
|---|---|
| `synapse/` (`SynapseConnectivityLong`) | `SAMPLE_N` (sampled) |
| `synapsefeatures/v1dd_1196_synapse_features/` (wide features) | `SAMPLE_N` |
| `synapsefeaturematrix/` (`SynapseFeatureMatrix` pointer) | 1 |

Sampled to `SAMPLE_N` rows and isolated under `DEMO_ROOT`; a production run drops
the sample cap and the `DEMO_ROOT` isolation once the `WriteSpec` entries above
are added. `synaptictargetlabel` is null for synapses without a label (LEFT join).